In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MHA(nn.Module):

    def __init__(self, input_dim, num_head):
        super().__init__()
        assert input_dim % num_head == 0
        self.input_dim = input_dim
        self.num_head = num_head
        self.d_k = input_dim // num_head
        self.wqkv = nn.Linear(input_dim, input_dim * 3)
        self.wo = nn.Linear(input_dim, input_dim)

    def forward(self, x, mask=None):
        B, L, D = x.shape
        qkv = self.wqkv(x)
        q, k, v = torch.chunk(qkv, 3, dim=-1)
        q = q.view(B, L, self.num_head, self.d_k).transpose(1, 2)
        k = k.view(B, L, self.num_head, self.d_k).transpose(1, 2)
        v = v.view(B, L, self.num_head, self.d_k).transpose(1, 2)
        attention = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(self.d_k)
        if mask is not None:
            attention = attention.masked_fill(mask == 0, -1e9)
        attention_scores = F.softmax(attention, dim=-1)
        context = torch.matmul(attention_scores, v)
        context = context.transpose(1, 2).contiguous().view(B, L, D)
        output = self.wo(context)
        return output
B, L, D = 10, 20, 36
x = torch.randn(B, L, D)
mha = MHA(D, 2)

print(mha(x).shape)

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MQA(nn.Module):

    def __init__(self, input_dim, num_head):
        super().__init__()
        assert input_dim % num_head == 0
        self.input_dim = input_dim
        self.num_head = num_head
        self.d_k = input_dim // num_head
        self.wq = nn.Linear(input_dim, input_dim)
        self.wkv = nn.Linear(input_dim, self.d_k * 2)
        self.wo = nn.Linear(input_dim, input_dim)

    def forward(self, x, mask=None):
        B, L, D = x.shape
        q = self.wq(x)
        kv = self.wkv(x)
        k, v = torch.chunk(kv, 2, dim=-1)
        q = q.view(B, L, self.num_head, self.d_k).transpose(1, 2)
        k = torch.unsqueeze(k, 1)   # (B, 1, L, d_k)
        v = torch.unsqueeze(v, 1)   # (B, 1, L, d_k)
        attention = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(self.d_k)
        if mask is not None:
            attention = attention.masked_fill(mask == 0, -1e9)
        attention_scores = F.softmax(attention, dim=-1)
        context = torch.matmul(attention_scores, v)
        context = context.transpose(1, 2).contiguous().view(B, L, D)
        output = self.wo(context)
        return output
B, L, D = 10, 20, 36
x = torch.randn(B, L, D)
mha = MQA(D, 2)

print(mha(x).shape)

torch.Size([10, 20, 36])


In [ ]:
class MQA(nn.Module):
    def __init__(self, input_dim, num_head):
        super().__init__()
        self.num_head = num_head
        self.d_k = input_dim // num_head
        self.wq  = nn.Linear(input_dim, input_dim)
        self.wkv = nn.Linear(input_dim, self.d_k * 2)
        self.wo = nn.Linear(input_dim, input_dim)
    
    def forward

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class GQA(nn.Module):

    def __init__(self, input_dim, q_num_head, kv_num_head):
        super().__init__()
        assert input_dim % q_num_head == 0
        assert q_num_head % kv_num_head == 0
        self.input_dim = input_dim
        self.q_num_head = q_num_head
        self.kv_num_head = kv_num_head
        self.d_k = input_dim // q_num_head
        self.num_q_group = q_num_head // kv_num_head

        self.wq = nn.Linear(input_dim, input_dim)
        self.wkv = nn.Linear(input_dim, self.d_k * kv_num_head * 2)
        self.wo = nn.Linear(input_dim, input_dim)

    def forward(self, x, mask=None):
        B, L, D = x.shape
        q = self.wq(x)
        q = q.view(B, L, self.q_num_head, self.d_k).transpose(1, 2).contiguous().view(B, self.kv_num_head, self.num_q_group, L, self.d_k)
        kv = self.wkv(x)
        k, v = torch.chunk(kv, 2, dim=-1)
        k = k.view(B, L, self.kv_num_head, self.d_k).transpose(1, 2)
        v = v.view(B, L, self.kv_num_head, self.d_k).transpose(1, 2)
        k = torch.unsqueeze(k, 2)   # (B, self.kv_num_head, 1, L, d_k)
        v = torch.unsqueeze(v, 2)   # (B, self.kv_num_head, 1, L, d_k)
        attention = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(self.d_k)
        if mask is not None:
            attention = attention.masked_fill(mask == 0, -1e9)
        attention_scores = F.softmax(attention, dim=-1)
        context = torch.matmul(attention_scores, v)
        context = context.view(B, self.q_num_head, L, self.d_k).transpose(1, 2).contiguous().view(B, L, D)
        output = self.wo(context)
        return output
B, L, D = 10, 20, 36
x = torch.randn(B, L, D)
mha = GQA(D, 4, 2)

print(mha(x).shape)

torch.Size([10, 20, 36])


In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, activation="gelu", norm="LN", dropout=0.1):        
        super().__init__()
        layer_dim = [input_dim] + hidden_dim + [output_dim]
        layers = []                   
        n = len(layer_dim)

        for i in range(n - 1):
            layers.append(nn.Linear(layer_dim[i], layer_dim[i + 1]))

            if i < n - 2:
                if norm == "LN":                                    
                    layers.append(nn.LayerNorm(layer_dim[i + 1]))  
                elif norm == "BN":                                 
                    layers.append(nn.BatchNorm1d(layer_dim[i + 1]))
                else:
                    raise ValueError(f"不支持的归一化: {norm}")

                if activation == "relu":
                    layers.append(nn.ReLU())
                elif activation == "tanh":
                    layers.append(nn.Tanh())
                elif activation == "sigmoid":
                    layers.append(nn.Sigmoid())
                elif activation == "gelu":                          
                    layers.append(nn.GELU())
                else:
                    raise ValueError(f"不支持的激活函数: {activation}")
                if dropout > 0:                                    
                    layers.append(nn.Dropout(dropout))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


# ===== 验证 =====
model = MLP(input_dim=64, hidden_dim=[128, 128], output_dim=10, norm="LN")
x = torch.randn(32, 64)
print(model(x).shape)   

torch.Size([32, 10])


In [32]:
import torch
def cross_entropy(logits, target):
    B, D = logits.shape
    logit_max = torch.max(logits, dim=-1)[0]
    logit_stable = logits - logit_max
    logit_log_sum_exp = torch.log(torch.sum(torch.exp(logit_stable), dim=-1))
    pos_logits = logit_stable[torch.arange(B), target]
    loss = - pos_logits + logit_log_sum_exp
    return loss.mean()
logits = torch.tensor([[1.0000, 0.4985, 0.6664, 0.2533],
                    [0.4985, 1.0000, 0.8408, 0.5431],
                    [0.6664, 0.8408, 1.0000, 0.8372],
                    [0.2533, 0.5431, 0.8372, 1.0000]])
target = torch.tensor([0, 1, 2, 3])
print(cross_entropy(logits, target))

tensor(1.1176)


In [ ]:
import torch
import torch.nn.functional as F
def BCE_loss(logits, label):
    loss = torch.clamp(logits, min=0) - logits * label + torch.log(1 + torch.exp(-torch.abs(logits)))
    return loss.mean()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
def infoNce(user_emb, item_emb, temperature=1.0):
    """
    user_emb: (B, D)
    item_emb: (B, D)
    """
    user_emb = F.normalize(user_emb, dim=-1)
    item_emb = F.normalize(item_emb, dim=-1)
    sim = torch.matmul(user_emb, item_emb.transpose(0, 1)) / temperature
    label = torch.arange(user_emb.size(0))
    loss = F.cross_entropy(sim, label)
    return loss.mean()

In [ ]:
import numpy as np
from collections import defaultdict

def _auc_single_user(user_scores, user_labels):
    """单用户 AUC，处理 tie（同分算 0.5）"""
    order = np.argsort(user_scores)
    sorted_scores = user_scores[order]
    sorted_labels = user_labels[order]

    n_pos = np.sum(sorted_labels == 1)
    n_neg = np.sum(sorted_labels == 0)

    cum_neg = 0
    correct_pairs = 0.0
    l, n = 0, len(sorted_labels)

    while l < n:
        r = l
        # 找到同分的一组
        while r < n and sorted_scores[r] == sorted_scores[l]:
            r += 1
        # [l, r) 为同分组
        group_pos = np.sum(sorted_labels[l:r] == 1)
        group_neg = np.sum(sorted_labels[l:r] == 0)
        # 同分组内正负对算 0.5，组前负样本算完全正确
        correct_pairs += group_pos * cum_neg + group_pos * group_neg * 0.5
        # 注意先计算正确的配对，再更新累计负样本，因为group_neg本组内和正样本预测分数相等的
        cum_neg += group_neg
        l = r

    return correct_pairs / (n_pos * n_neg)


def gauc_rank(user_ids, labels, scores, weight_type='impression'):

    # 去掉细节，GAUC 的本质只有两步：
    # GAUC = Σ(用户i的AUC × 用户i的权重) / Σ(用户i的权重)

    user_ids = np.array(user_ids)
    labels   = np.array(labels)
    scores   = np.array(scores)

    user_sample_dict = defaultdict(list)
    for idx, uid in enumerate(user_ids):
        user_sample_dict[uid].append(idx)

    user_auc_dict      = {}
    total_weighted_auc = 0.0
    total_weight       = 0.0

    for uid, indices in user_sample_dict.items():
        user_labels = labels[indices]
        user_scores = scores[indices]

        n_pos = np.sum(user_labels == 1)
        n_neg = np.sum(user_labels == 0)

        # 无正样本或者负样本的用户，无法计算auc，直接过滤
        if n_pos == 0 or n_neg == 0:
            continue

        user_auc = _auc_single_user(user_scores, user_labels)
        user_auc_dict[uid] = user_auc

        if weight_type == 'impression':
            weight = len(indices)
        elif weight_type == 'uniform':
            weight = 1.0
        else:
            raise ValueError(f"Unknown weight_type: {weight_type}")

        total_weighted_auc += user_auc * weight
        total_weight       += weight
    
    if total_weight == 0:
        raise ValueError("No valid user found for AUC computation.")
    
    gauc = total_weighted_auc / total_weight

    return gauc, user_auc_dict


label   = [0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0]
q       = [0.1, 0.9, 0.2, 0.8, 1, 0.2, 0.3, 0.9, 0.7, 0.9, 0.7]
user_id = [1,   1,   1,   1,   2, 2,   2,   2,   3,   3,   3  ]
gauc_rank(user_id, label, q, weight_type='impression')

## 逻辑回归

In [ ]:
import torch

def sigmoid(logits):
    return 1 / (1 + torch.exp(-logits))

def softmax(logits):
    max_logits = torch.max(logits, dim=1, keepdim=True)[0]
    logits_stable = logits - max_logits
    exp_logits_stable = torch.exp(logits_stable)
    sum_exp = torch.sum(exp_logits_stable, dim=-1, keepdim=True)
    return exp_logits_stable / sum_exp

def BCEloss(logits, y):
    """logits.shape: [B, 1], y.shape: [B, 1]"""
    eps = 1e-9
    p = sigmoid(logits)
    loss = -(y * torch.log(p + eps) + (1 - y) * torch.log(1 - p + eps))
    return loss.mean()

def CEloss(logits, y):
    """logits.shape: [B, D], y.shape: [B, D](one-hot/soft)"""
    eps = 1e-9
    sm = softmax(logits)
    loss_per_element = -y * torch.log(sm + eps)
    return loss_per_element.sum(dim=-1).mean()

In [8]:
import math

def sigmoid(x):
    return 1 / (1+math.exp(-x))

def solve_by_gradient_desent(y):
    x = 0.0
    lr = 1e-3
    max_iter = 1000
    max_loss = 1e-6
    for _ in range(max_iter):
        sig_x = sigmoid(x)
        loss = (sig_x - y)**2
        grad_x = 2 * (sig_x - y) * sig_x * (1-sig_x)
        x -= lr*grad_x
        if loss < max_loss:
            break
    return x

solve_by_gradient_desent(0.5)

0.0

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(42)

# ================================================================
# 1. 合成数据：4 个任务，其中 2 个干净、2 个噪声大
# ================================================================
N, D = 20000, 16
X = torch.randn(N, D)

# --- 任务组 A（类比一跳组）：两个二分类任务 ---
logit_a1 = (X[:, :4] * torch.tensor([1.0, -1.0, 0.5, -0.5])).sum(1)
y_a1 = (torch.sigmoid(logit_a1) > 0.5).float()             # A1：干净

logit_a2 = (X[:, 4:8] * torch.tensor([1.0, 1.0, -1.0, -1.0])).sum(1)
y_a2 = (torch.sigmoid(logit_a2) > 0.5).float()
flip = torch.rand(N) < 0.30
y_a2 = torch.where(flip, 1 - y_a2, y_a2)                   # A2：30% 标签翻转，噪声大

# --- 任务组 B（类比二跳组）：两个回归任务 ---
y_b1 = (X[:, 8:12] * torch.tensor([0.5, 0.3, -0.4, 0.2])).sum(1)
y_b1 += 0.1 * torch.randn(N)                               # B1：小噪声

y_b2 = (X[:, 12:16] * torch.tensor([0.4, -0.3, 0.5, -0.2])).sum(1)
y_b2 += 1.0 * torch.randn(N)                               # B2：大噪声

# ================================================================
# 2. 模型：embedding 全局共享，组内 tower 独立
# ================================================================
class MultiTaskModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.shared_embed = nn.Sequential(nn.Linear(D, 32), nn.ReLU())  # 全局共享
        self.tower_A = nn.Sequential(nn.Linear(32, 16), nn.ReLU())      # 组 A 独占
        self.tower_B = nn.Sequential(nn.Linear(32, 16), nn.ReLU())      # 组 B 独占
        self.head_a1 = nn.Linear(16, 1)
        self.head_a2 = nn.Linear(16, 1)
        self.head_b1 = nn.Linear(16, 1)
        self.head_b2 = nn.Linear(16, 1)

    def forward(self, x):
        z = self.shared_embed(x)
        a, b = self.tower_A(z), self.tower_B(z)
        return {
            "a1": self.head_a1(a).squeeze(-1),
            "a2": self.head_a2(a).squeeze(-1),
            "b1": self.head_b1(b).squeeze(-1),
            "b2": self.head_b2(b).squeeze(-1),
        }

# ================================================================
# 3. UW 损失模块（关键！）
# ================================================================
class UncertaintyWeightedLoss(nn.Module):
    """
    对一组任务做 Uncertainty Weighting。
    
    核心：每个任务注册一个 nn.Parameter（log_sigma），
    它和模型权重一起 backward、一起更新。
    """
    def __init__(self, task_names):
        super().__init__()
        # 为每个任务创建一个标量参数，初始化为 0（即 σ=1）
        self.log_sigmas = nn.ParameterDict({
            name: nn.Parameter(torch.zeros(1)) for name in task_names
        })

    def forward(self, losses: dict):
        total = 0.0
        for name, loss in losses.items():
            log_sigma = self.log_sigmas[name]
            # 1/(2σ²) · L + log σ
            # 用 exp(-2·log_sigma) 计算 1/σ²，保证数值稳定 & σ > 0
            precision = torch.exp(-2.0 * log_sigma)
            total = total + 0.5 * precision * loss + log_sigma
        return total.squeeze()

    def sigmas(self):
        return {n: torch.exp(p).item() for n, p in self.log_sigmas.items()}


# ================================================================
# 4. 训练对比
# ================================================================
def train(use_uw: bool, n_epochs=30, bs=512):
    model = MultiTaskModel()
    bce, mse = nn.BCEWithLogitsLoss(), nn.MSELoss()

    if use_uw:
        # 组 A 一个 UW 模块，组 B 一个 UW 模块（组内 UW，组间等权）
        uw_A = UncertaintyWeightedLoss(["a1", "a2"])
        uw_B = UncertaintyWeightedLoss(["b1", "b2"])
        params = list(model.parameters()) + list(uw_A.parameters()) + list(uw_B.parameters())
    else:
        params = list(model.parameters())

    opt = optim.Adam(params, lr=1e-3)
    n_batch = N // bs

    for ep in range(n_epochs):
        perm = torch.randperm(N)
        ep_loss = {"a1": 0, "a2": 0, "b1": 0, "b2": 0}

        for i in range(n_batch):
            idx = perm[i*bs:(i+1)*bs]
            p = model(X[idx])
            raw = {
                "a1": bce(p["a1"], y_a1[idx]),
                "a2": bce(p["a2"], y_a2[idx]),
                "b1": mse(p["b1"], y_b1[idx]),
                "b2": mse(p["b2"], y_b2[idx]),
            }
            if use_uw:
                # 组内 UW，组间等权
                total = uw_A({"a1": raw["a1"], "a2": raw["a2"]}) \
                      + uw_B({"b1": raw["b1"], "b2": raw["b2"]})
            else:
                total = sum(raw.values())  # baseline: 纯等权

            opt.zero_grad()
            total.backward()
            opt.step()

            for k in ep_loss:
                ep_loss[k] += raw[k].item() / n_batch

        if (ep + 1) % 5 == 0:
            msg = f"Epoch {ep+1:2d} | " + " ".join(
                f"{k}={v:.3f}" for k, v in ep_loss.items())
            if use_uw:
                s = {**uw_A.sigmas(), **uw_B.sigmas()}
                msg += " | σ: " + " ".join(f"{k}={v:.3f}" for k, v in s.items())
            print(msg)

print("=== 等权求和（baseline）===")
train(use_uw=False)

print("\n=== Uncertainty Weighting ===")
train(use_uw=True)

=== 等权求和（baseline）===
Epoch  5 | a1=0.181 a2=0.650 b1=0.026 b2=1.019
Epoch 10 | a1=0.069 a2=0.642 b1=0.017 b2=0.999
Epoch 15 | a1=0.045 a2=0.639 b1=0.016 b2=0.994
Epoch 20 | a1=0.034 a2=0.638 b1=0.016 b2=0.993
Epoch 25 | a1=0.028 a2=0.638 b1=0.015 b2=0.992
Epoch 30 | a1=0.024 a2=0.636 b1=0.015 b2=0.991

=== Uncertainty Weighting ===
Epoch  5 | a1=0.149 a2=0.651 b1=0.027 b2=1.013 | σ: a1=0.785 a2=0.853 b1=0.786 b2=1.046
Epoch 10 | a1=0.050 a2=0.642 b1=0.015 b2=1.000 | σ: a1=0.613 a2=0.806 b1=0.630 b2=1.008
Epoch 15 | a1=0.029 a2=0.640 b1=0.013 b2=0.997 | σ: a1=0.491 a2=0.800 b1=0.512 b2=0.999
Epoch 20 | a1=0.021 a2=0.638 b1=0.013 b2=0.997 | σ: a1=0.397 a2=0.799 b1=0.420 b2=0.999
Epoch 25 | a1=0.015 a2=0.637 b1=0.012 b2=0.996 | σ: a1=0.324 a2=0.798 b1=0.346 b2=0.997
Epoch 30 | a1=0.012 a2=0.635 b1=0.011 b2=0.995 | σ: a1=0.266 a2=0.797 b1=0.286 b2=0.997


In [8]:
def find_length_str(nums):
    n = len(nums)
    dp = [1] * n
    prev_index = [-1] * n
    for i in range(1, n):
        for j in range(i):
            if nums[i] > nums[j] and dp[i] < dp[j] + 1:
                dp[i] = dp[j] + 1
                prev_index[i] = j
    max_len = max(dp)
    end_index = dp.index(max_len)
    path = []
    cur_index = end_index
    while cur_index != -1:
        path.append(nums[cur_index])
        cur_index = prev_index[cur_index]
    return list(reversed(path))

nums = [20, 10, 14, 18, 9, 30]
print(find_length_str(nums))

[10, 14, 18, 30]
